In [21]:
!pip install openmeteo-requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.0/711.0 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.8/622.8 kB 1.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 2.9 MB/s eta 0:00:00a 0:00:01m


In [22]:
from datetime import datetime
import openmeteo_requests


class IncreaseSpeed:
  
    def __init__(self, current_speed: int, max_speed: int, step=10):
        self.current_speed = current_speed
        self.max_speed = max_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed >= self.max_speed:
            raise StopIteration

        self.current_speed = min(self.current_speed + self.step, self.max_speed)
        return self.current_speed


class DecreaseSpeed:

    def __init__(self, current_speed: int, min_speed: int = 0, step=10):
        self.current_speed = current_speed
        self.min_speed = min_speed
        self.step = step

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_speed <= self.min_speed:
            raise StopIteration

        self.current_speed = max(self.current_speed - self.step, self.min_speed)
        return self.current_speed


class Car:
    cars_on_road = 0 #class variable

    def __init__(self, max_speed: int, current_speed=0): #create the new object of Car
        self.max_speed = max_speed
        self.current_speed = current_speed
        if current_speed > 0:
            self.state = True
        else:
            self.state = False #car confitions - if > 0 True - car on road, False - parking

        if self.state: #if its True, its on raod, so +1 car
            Car.cars_on_road += 1

    def accelerate(self, upper_border=None, step=10): #increase speed method
        old_speed = self.current_speed

        if upper_border is None: #if no data about desired speed, increase it by one step but not allow to exceed the max speed
            upper_border = min(self.current_speed + step, self.max_speed) 
        else:
            upper_border = min(upper_border, self.max_speed)

        if upper_border < self.current_speed: #cannot accelerate downwards
            upper_border = self.current_speed

        increaser = IncreaseSpeed(self.current_speed, upper_border, step)

        for new_speed in increaser: #iterate
            print(f"INFO: Speed increases by {step}")
            self.current_speed = new_speed #renew speed for car

        if not self.state and self.current_speed > 0: #check wass the car on road or not - if not and started to move
            self.state = True  #its on road now
            Car.cars_on_road += 1 #add it

        print(
            f"INFO: The speed of this car has been increased from "
            f"{old_speed} to {self.current_speed}"
        )

    def brake(self, lower_border=None, step=10): #decrease speed, the same as in accelerate
        old_speed = self.current_speed

        if lower_border is None:
            lower_border = max(self.current_speed - step, 0)
        else:
            lower_border = max(lower_border, 0) #speed cannot be negative

        if lower_border > self.current_speed:
            lower_border = self.current_speed

        decreaser = DecreaseSpeed(self.current_speed, lower_border, step)

        for new_speed in decreaser:
            print(f"INFO: Speed decreases by {step}")
            self.current_speed = new_speed

        print(
            f"INFO: The speed of this car has been decreased from "
            f"{old_speed} to {self.current_speed}"
        )

    def parking(self): #send car to parking 
        if not self.state: #in not on the road (on parking), do nothing
            print("INFO: The car is already in parking")
            return

        self.brake(0) #if on road, decrease speed to 0
        self.state = False #parking
        Car.cars_on_road -= 1 #remove car from count
        print("Parking the car...")

    @classmethod
    def total_cars(Car): #work wit hclass, not the object 
        return Car.cars_on_road

    @staticmethod
    def show_weather(): #show weather with any car, anytime, doesnt link to Car/objects
        openmeteo = openmeteo_requests.Client() #all from snippet
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": 59.9386,
            "longitude": 30.3141,
            "current": [
                "temperature_2m",
                "apparent_temperature",
                "rain",
                "wind_speed_10m",
            ],
            "wind_speed_unit": "ms",
            "timezone": "Europe/Moscow",
        }

        response = openmeteo.weather_api(url, params=params)[0]

        current = response.Current()
        current_temperature_2m = current.Variables(0).Value()
        current_apparent_temperature = current.Variables(1).Value()
        current_rain = current.Variables(2).Value()
        current_wind_speed_10m = current.Variables(3).Value()

        print(f"Current temperature: {round(current_temperature_2m, 0)} C")
        print(f"Current apparent_temperature: {round(current_apparent_temperature, 0)} C")
        print(f"Current rain: {current_rain} mm")
        print(f"Current wind_speed: {round(current_wind_speed_10m, 1)} m/s")

In [23]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [24]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [25]:
car2.accelerate(50)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50


In [26]:
print("Speed of car 1:", car1.current_speed)

Speed of car 1: 100


In [27]:
print("Speed of car 2:", car2.current_speed)

Speed of car 2: 50


In [28]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [29]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [30]:
car3.accelerate(80)


INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 80


In [31]:
car3.show_weather()
print("Total cars on road:", Car.total_cars())

Current temperature: 4.0 C
Current apparent_temperature: 0.0 C
Current rain: 0.0 mm
Current wind_speed: 2.9 m/s
Total cars on road: 2


In [32]:
car2.accelerate(10)
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3


In [33]:
Car.show_weather()

Current temperature: 4.0 C
Current apparent_temperature: 0.0 C
Current rain: 0.0 mm
Current wind_speed: 2.9 m/s
